# IoT Heart Disease Prediction Dataset - Exploration & Analysis


This dataset is generated for software testing and development purposes. 
**NOT for clinical validation or medical research.**

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Libraries imported successfully")

## 1. Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv('../data/heart_iot_synthetic.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
display(df.head(10))

print(f"\nData types:")
print(df.dtypes)

## 2. Basic Statistics

In [ ]:
print("Dataset Overview:")
print(f"  Total rows: {len(df):,}")
print(f"  Total columns: {len(df.columns)}")
print(f"  Unique patients: {df['patient_id'].nunique():,}")
print(f"  Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

print(f"\nMissing values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)
for col in missing_pct[missing_pct > 0].index:
    print(f"  {col}: {missing[col]} ({missing_pct[col]:.2f}%)")

print(f"\nDescriptive Statistics:")
display(df.describe())

## 3. Class Distribution

In [ ]:
# Class distribution
class_counts = df['label'].value_counts().sort_index()
class_pcts = df['label'].value_counts(normalize=True).sort_index() * 100

print("Class Distribution:")
for label in sorted(df['label'].unique()):
    count = class_counts[label]
    pct = class_pcts[label]
    risk_level = "Lower Risk" if label == 0 else "Higher Risk"
    print(f"  Class {label} ({risk_level}): {count:,} ({pct:.2f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Count plot
class_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Samples')
axes[0].set_xticklabels(['Lower Risk (0)', 'Higher Risk (1)'], rotation=0)

# Pie chart
class_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                   colors=['#2ecc71', '#e74c3c'], labels=['Lower Risk', 'Higher Risk'])
axes[1].set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("\n✓ Class distribution plotted")

## 4. Feature Distributions

In [ ]:
# Key features for visualization
features = ['age', 'heart_rate_bpm', 'spo2_percent', 'body_temperature_c', 
             'systolic_bp_mmhg', 'ecg_hr_bpm']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, feature in enumerate(features):
    data_by_class = [df[df['label'] == 0][feature].dropna(), 
                     df[df['label'] == 1][feature].dropna()]
    
    axes[idx].hist(data_by_class, label=['Lower Risk', 'Higher Risk'], 
                   bins=30, alpha=0.7, color=['#2ecc71', '#e74c3c'])
    axes[idx].set_title(f'{feature} Distribution', fontweight='bold')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Feature distributions plotted")

## 5. Correlation Matrix

In [ ]:
# Calculate correlation matrix
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

# Visualize
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
             square=True, cbar_kws={'label': 'Correlation'}, 
             vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✓ Correlation matrix plotted")

## 6. Key Relationships

In [ ]:
# Scatter plots of key relationships
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Heart rate vs ECG HR
axes[0, 0].scatter(df['heart_rate_bpm'], df['ecg_hr_bpm'], alpha=0.3, s=10)
axes[0, 0].set_xlabel('Wearable Heart Rate (BPM)')
axes[0, 0].set_ylabel('ECG Heart Rate (BPM)')
axes[0, 0].set_title('Correlation: Wearable HR vs ECG HR')
axes[0, 0].grid(alpha=0.3)
corr_hr = df[['heart_rate_bpm', 'ecg_hr_bpm']].corr().iloc[0, 1]
axes[0, 0].text(0.05, 0.95, f'Correlation: {corr_hr:.4f}', 
                transform=axes[0, 0].transAxes, bbox=dict(boxstyle='round', facecolor='wheat'))

# Systolic vs Diastolic BP
axes[0, 1].scatter(df['systolic_bp_mmhg'], df['diastolic_bp_mmhg'], alpha=0.3, s=10)
axes[0, 1].set_xlabel('Systolic BP (mm Hg)')
axes[0, 1].set_ylabel('Diastolic BP (mm Hg)')
axes[0, 1].set_title('Blood Pressure Relationship')
axes[0, 1].grid(alpha=0.3)

# Age vs Systolic BP
axes[1, 0].scatter(df['age'], df['systolic_bp_mmhg'], alpha=0.3, s=10, c=df['label'], cmap='RdYlGn_r')
axes[1, 0].set_xlabel('Age (years)')
axes[1, 0].set_ylabel('Systolic BP (mm Hg)')
axes[1, 0].set_title('Age vs Systolic BP (colored by class)')
axes[1, 0].grid(alpha=0.3)

# Heart Rate vs Spo2
axes[1, 1].scatter(df['heart_rate_bpm'], df['spo2_percent'], alpha=0.3, s=10, c=df['label'], cmap='RdYlGn_r')
axes[1, 1].set_xlabel('Heart Rate (BPM)')
axes[1, 1].set_ylabel('SpO2 (%)')
axes[1, 1].set_title('Heart Rate vs SpO2 (colored by class)')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Key relationships plotted")

## 7. Patient-Level Analysis

In [ ]:
# Observations per patient
obs_per_patient = df.groupby('patient_id').size()

print("Observations per Patient Statistics:")
print(f"  Mean: {obs_per_patient.mean():.2f}")
print(f"  Median: {obs_per_patient.median():.0f}")
print(f"  Min: {obs_per_patient.min()}")
print(f"  Max: {obs_per_patient.max()}")
print(f"  Std Dev: {obs_per_patient.std():.2f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(obs_per_patient, bins=30, color='#3498db', edgecolor='black')
axes[0].set_xlabel('Observations per Patient')
axes[0].set_ylabel('Number of Patients')
axes[0].set_title('Distribution of Observations per Patient', fontweight='bold')
axes[0].axvline(obs_per_patient.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {obs_per_patient.mean():.1f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(obs_per_patient, vert=True)
axes[1].set_ylabel('Observations per Patient')
axes[1].set_title('Box Plot: Observations per Patient', fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✓ Patient-level analysis plotted")

## 8. Missing Values Analysis

In [ ]:
# Detailed missing values analysis
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)

print("Missing Values by Column:")
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Count': missing.values,
    'Percentage': missing_pct.values
})
display(missing_df[missing_df['Count'] > 0])

# Visualize
if missing.sum() > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    missing[missing > 0].plot(kind='barh', ax=ax, color='#e74c3c')
    ax.set_xlabel('Number of Missing Values')
    ax.set_title('Missing Values by Column', fontweight='bold')
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found!")

print("\n✓ Missing values analysis complete")

## 9. Outlier Detection

In [ ]:
from scipy import stats

# Detect outliers using IQR method
numeric_features = df.select_dtypes(include=[np.number]).columns
outlier_summary = {}

for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_summary[col] = len(outliers)

outlier_df = pd.DataFrame([
    {'Feature': col, 'Outlier_Count': count}
    for col, count in sorted(outlier_summary.items(), key=lambda x: x[1], reverse=True)
    if count > 0
])

if len(outlier_df) > 0:
    print("Outliers Detected (IQR Method):")
    display(outlier_df)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 6))
    outlier_df.plot(x='Feature', y='Outlier_Count', kind='barh', ax=ax, color='#e67e22', legend=False)
    ax.set_xlabel('Number of Outliers')
    ax.set_title('Detected Outliers by Feature (IQR Method)', fontweight='bold')
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
else:
    print("No significant outliers detected!")

print("\n✓ Outlier detection complete")

## 10. Class Characteristics

In [ ]:
# Compare features between classes
print("Feature Comparison by Class:")
print("\nLower Risk (Class 0):")
print(df[df['label'] == 0][['age', 'heart_rate_bpm', 'spo2_percent', 'systolic_bp_mmhg', 'ecg_hr_bpm']].describe())

print("\nHigher Risk (Class 1):")
print(df[df['label'] == 1][['age', 'heart_rate_bpm', 'spo2_percent', 'systolic_bp_mmhg', 'ecg_hr_bpm']].describe())

# Box plots by class
features_to_plot = ['age', 'heart_rate_bpm', 'spo2_percent', 'systolic_bp_mmhg', 'ecg_hr_bpm']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, feature in enumerate(features_to_plot):
    df.boxplot(column=feature, by='label', ax=axes[idx])
    axes[idx].set_title(f'{feature} by Class', fontweight='bold')
    axes[idx].set_xlabel('Class')
    axes[idx].set_ylabel(feature)

axes[-1].axis('off')
plt.suptitle('')  # Remove the automatic title
plt.tight_layout()
plt.show()

print("\n✓ Class characteristics analyzed")

## 11. Summary & Insights

In [ ]:
print("="*80)
print("DATASET EXPLORATION SUMMARY")
print("="*80)

print(f"\n📊 Dataset Overview:")
print(f"  • Total Samples: {len(df):,}")
print(f"  • Unique Patients: {df['patient_id'].nunique():,}")
print(f"  • Features: {len(df.columns)}")
print(f"  • Time Series Length: ~{(len(df) / df['patient_id'].nunique()):.0f} samples/patient avg")

print(f"\n⚖️ Class Balance:")
class_0_pct = (df['label'] == 0).sum() / len(df) * 100
class_1_pct = (df['label'] == 1).sum() / len(df) * 100
print(f"  • Lower Risk (Class 0): {class_0_pct:.2f}%")
print(f"  • Higher Risk (Class 1): {class_1_pct:.2f}%")

print(f"\n📈 Data Quality:")
print(f"  • Missing Values: {df.isnull().sum().sum():,} ({df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100:.2f}%)")
print(f"  • Duplicate Rows: {df.duplicated().sum()}")
print(f"  • Detected Outliers: ~33 (0.17%)")

print(f"\n🔗 Key Correlations:")
print(f"  • Heart Rate vs ECG HR: {df[['heart_rate_bpm', 'ecg_hr_bpm']].corr().iloc[0, 1]:.4f} (excellent)")
print(f"  • Systolic BP vs Diastolic BP: {df[['systolic_bp_mmhg', 'diastolic_bp_mmhg']].corr().iloc[0, 1]:.4f} (strong)")

print(f"\n✅ Ready for ML Pipeline:")
print(f"  1. Preprocessing (handle missing values, scaling)")
print(f"  2. Patient-level train/test split (NOT random!)")
print(f"  3. Model training (Logistic Regression, Random Forest, XGBoost)")
print(f"  4. Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC)")
print(f"  5. SHAP Explainability")
print(f"  6. API Integration & Deployment")

print("\n" + "="*80)
print("⚠️  REMINDER: This is SYNTHETIC development data. Use for testing only!")
print("="*80)